# Day 30 — Week 5–6 Capstone: Full Analytics Report
**Estimated time:** 3 hours  
**Learning objectives:**
1. Synthesise all Week 5–6 skills into a single, presentation-ready analytics report
2. Use both SQL (sqlite3) and Python (pandas) appropriately for each section
3. Produce a self-contained business review package

---
> **Self-assessment question:** After completing this notebook, ask yourself: *"Could I present each section of this to a senior stakeholder in a real meeting?"* If not, identify which section needs more work before moving to Week 7.


## 0. Setup — Load Everything

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from textwrap import dedent

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

from analytics_bootcamp.config import RAW_DATA_DIR as DATA
TODAY = pd.Timestamp('2026-04-01')
REPORT_PERIOD = 202412

con = duckdb.connect()
dfs = {}
for tname, fname in [
    ('materials','materials_inventory.csv'),
    ('sales_orders','sales_orders.csv'),
    ('cost_centers','cost_center_actuals.csv'),
    ('hr','hr_headcount.csv'),
    ('bw_kpis','bw_sales_kpis.csv'),
]:
    df = pd.read_csv(DATA / fname)
    con.register(tname, df)
    dfs[tname] = df

def q(sql): return con.sql(sql).df()

print('Capstone ready. All datasets loaded.')


## Section 1 — Executive Summary (5 KPIs as Text Output)

In [ ]:
# YOUR SOLUTION
# This section should take ~20 min. Produce 5 executive KPIs as clean print output.
# Think: what would appear in the 'Key Highlights' box of a management report?

bw = dfs['bw_kpis']
so = dfs['sales_orders']
mat = dfs['materials']
hr = dfs['hr']
cc = dfs['cost_centers']

cy = 2025
py = 2024

cy_rev = bw[bw['CAL_YEAR_MONTH']//100 == cy]['REVENUE'].sum()
py_rev = bw[bw['CAL_YEAR_MONTH']//100 == py]['REVENUE'].sum()
yoy_pct = (cy_rev - py_rev) / py_rev * 100

cy_gm_pct = (bw[bw['CAL_YEAR_MONTH']//100==cy]['GROSS_MARGIN'].sum() /
             bw[bw['CAL_YEAR_MONTH']//100==cy]['REVENUE'].sum() * 100)

open_backlog = so[so['STATUS']=='OPEN']['NETWR'].sum()

mat['INV_VALUE'] = mat['LABST'] * mat['STPRS']
mat['DAYS_OLD'] = (TODAY - pd.to_datetime(mat['LAST_MOVEMENT_DATE'], errors='coerce')).dt.days.fillna(999)
dead_value = mat[mat['DAYS_OLD'] > 180]['INV_VALUE'].sum()

active = hr[hr['TERM_DATE'].isna()]
total_salary_cost = active['SALARY'].sum()

# YTD cost variance
cc['VAR'] = cc['ACTUAL_AMT'] - cc['PLAN_AMT']
ytd_var = cc[cc['GJAHR']==2024]['VAR'].sum()

print('=' * 60)
print('         EXECUTIVE SUMMARY — BUSINESS REVIEW')
print('=' * 60)
print(f'  1. Revenue (CY):        ${cy_rev:>12,.0f}  ({yoy_pct:+.1f}% YoY)')
print(f'  2. Gross Margin %:       {cy_gm_pct:>10.1f}%')
print(f'  3. Open Order Backlog:  ${open_backlog:>12,.0f}')
print(f'  4. Dead Inventory ($):  ${dead_value:>12,.0f}  (>180d no movement)')
print(f'  5. YTD Cost Variance:   ${ytd_var:>12,.0f}  (+ = over budget)')
print('=' * 60)

## Section 2 — Revenue Performance vs Plan with Trend

In [ ]:
# YOUR SOLUTION
# SQL for the heavy aggregation; pandas/matplotlib for the chart

sql_rev_plan = '''
WITH monthly AS (
    SELECT CAL_YEAR_MONTH,
           SUM(REVENUE) AS actual_rev,
           SUM(COGS) AS cogs
    FROM bw_kpis
    WHERE CAL_YEAR_MONTH >= 202401
    GROUP BY CAL_YEAR_MONTH
),
prev_year AS (
    SELECT CAL_YEAR_MONTH + 100 AS cy_ym,
           SUM(REVENUE) AS py_rev
    FROM bw_kpis
    WHERE CAL_YEAR_MONTH >= 202301 AND CAL_YEAR_MONTH <= 202312
    GROUP BY CAL_YEAR_MONTH
)
SELECT m.CAL_YEAR_MONTH,
       ROUND(m.actual_rev, 0) AS Revenue,
       ROUND(p.py_rev, 0) AS Prior_Year,
       ROUND(m.actual_rev - p.py_rev, 0) AS YoY_Change,
       ROUND((m.actual_rev - p.py_rev)/NULLIF(p.py_rev,0)*100, 1) AS YoY_Pct
FROM monthly m
LEFT JOIN prev_year p ON m.CAL_YEAR_MONTH = p.cy_ym
ORDER BY m.CAL_YEAR_MONTH
'''

rev_df = q(sql_rev_plan)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rev_df.index, rev_df['Revenue']/1e6, marker='o', color='#2980b9', linewidth=2, label='2025 Revenue')
ax.plot(rev_df.index, rev_df['Prior_Year']/1e6, marker='s', color='#95a5a6',
        linewidth=2, linestyle='--', label='2024 Revenue')
ax.fill_between(rev_df.index, rev_df['Revenue']/1e6, rev_df['Prior_Year']/1e6,
                alpha=0.1, color='blue')
ax.set_xticks(rev_df.index)
ax.set_xticklabels([str(p) for p in rev_df['CAL_YEAR_MONTH']], rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}M'))
ax.set_title('Monthly Revenue: 2025 vs 2024', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('day30_revenue_trend.png', dpi=120, bbox_inches='tight')
plt.show()
print(rev_df.to_string(index=False))

## Section 3 — Inventory Health Summary

In [ ]:
# YOUR SOLUTION
mat = dfs['materials'].copy()
mat['INV_VALUE'] = mat['LABST'] * mat['STPRS']
mat['DAYS_OLD'] = (TODAY - pd.to_datetime(mat['LAST_MOVEMENT_DATE'], errors='coerce')).dt.days.fillna(999).astype(int)

bins = [0, 30, 90, 180, 9999]
labels = ['Current', 'Slow', 'Stagnant', 'Dead']
mat['BUCKET'] = pd.cut(mat['DAYS_OLD'], bins=bins, labels=labels)

inv_summary = mat.groupby(['WERKS', 'BUCKET'], observed=True).agg(
    Items=('MATNR','count'), Total_Value=('INV_VALUE','sum')
).reset_index()

total_inv = mat['INV_VALUE'].sum()
dead_inv = mat[mat['BUCKET']=='Dead']['INV_VALUE'].sum()
dead_pct = dead_inv / total_inv * 100

print(f'Total Inventory Value:  ${total_inv:,.0f}')
print(f'Dead Stock Value:        ${dead_inv:,.0f} ({dead_pct:.1f}% of total)')
print()
print(inv_summary.pivot_table(index='BUCKET', columns='WERKS', values='Total_Value',
                               aggfunc='sum', fill_value=0).applymap(lambda x: f'${x:,.0f}'))

## Section 4 — Workforce Cost Analysis

In [ ]:
# YOUR SOLUTION
hr = dfs['hr'].copy()
cc = dfs['cost_centers'].copy()

active = hr[hr['TERM_DATE'].isna()].copy()

# Salary cost by org unit
org_cost = active.groupby('ORGTX').agg(HC=('PERNR','count'), Salary=('SALARY','sum')).reset_index()
org_cost['Avg_Salary'] = org_cost['Salary'] / org_cost['HC']
org_cost = org_cost.sort_values('Salary', ascending=False)

# Terminations (last 12m)
hr['TERM_DATE_DT'] = pd.to_datetime(hr['TERM_DATE'], errors='coerce')
hr['HIRE_DATE_DT'] = pd.to_datetime(hr['HIRE_DATE'], errors='coerce')
terms_12m = hr[(hr['TERM_DATE_DT'].notna()) &
               (hr['TERM_DATE_DT'] >= TODAY - pd.DateOffset(months=12))].shape[0]
avg_hc = len(active)
turnover = terms_12m / avg_hc * 100

print(f'Active Headcount:   {avg_hc}')
print(f'Total Salary Cost:  ${active["SALARY"].sum():,.0f}')
print(f'Annualized Turnover:{turnover:.1f}%')
print()
print('=== Cost by Org Unit (Top 8) ===')
print(org_cost.head(8).to_string(index=False))

## Section 5 — Top Risks and Opportunities (Data-Driven Narrative)

In [ ]:
# YOUR SOLUTION
# Write data-driven risk/opportunity statements
# Each must reference an actual number from the analysis above

risks = [
    f'Dead Stock Risk: ${dead_inv:,.0f} ({dead_pct:.1f}% of inventory) has had no movement in 180+ days. '
    f'Write-off exposure if not repositioned within Q2.',

    f'Open Order Backlog: ${open_backlog:,.0f} in open orders; '
    f'{(pd.to_datetime(so[so["STATUS"]=="OPEN"]["ERDAT"], errors="coerce").apply(lambda d: (TODAY-d).days if pd.notna(d) else 999) > 60).sum()} orders are >60 days old. '
    f'Revenue recognition risk if delivery SLAs are breached.',

    f'Cost Overrun: YTD cost variance is ${ytd_var:+,.0f}. '
    f'If full-year trend continues, annual overrun is projected at ${ytd_var/9*12:+,.0f}.',
]

opportunities = [
    f'YoY Revenue Growth: {yoy_pct:+.1f}% growth signals demand strength. '
    f'Opportunity: accelerate top-performing regions.',

    f'Gross Margin: {cy_gm_pct:.1f}% — if discount rate is reduced by 1pp, '
    f'estimated margin improvement: ${bw[bw["CAL_YEAR_MONTH"]//100==cy]["DISCOUNT_AMT"].sum()/100:,.0f}.',
]

print('╔══════════════════════════════════════════════════════╗')
print('║              RISKS AND OPPORTUNITIES                 ║')
print('╠══════════════════════════════════════════════════════╣')
print('RISKS:')
for i, r in enumerate(risks, 1):
    import textwrap
    wrapped = textwrap.fill(f'{i}. {r}', width=70, subsequent_indent='   ')
    print(wrapped)
    print()
print('OPPORTUNITIES:')
for i, o in enumerate(opportunities, 1):
    wrapped = textwrap.fill(f'{i}. {o}', width=70, subsequent_indent='   ')
    print(wrapped)
print('╚══════════════════════════════════════════════════════╝')

## Self-Assessment

Rate yourself on each section:
- Section 1 Executive Summary: did you get all 5 KPIs correct with no errors?
- Section 2 Revenue Trend: does the chart look presentation-ready?
- Section 3 Inventory: did you correctly calculate dead stock at the plant level?
- Section 4 Workforce: did you join HR to CO data correctly?
- Section 5 Narrative: are your risk statements data-referenced and specific?

**If any section scored "needs work" — go back to the relevant day and repeat the exercise before Week 7.**

## Daily Prompt
**Final reflection (write in a new markdown cell):**

1. List 3 things you built this week that you could reuse in a real client engagement.
2. What single concept from Week 5–6 is still not fully clear? Name it specifically.
3. If a stakeholder asked you to present one of these analyses tomorrow, which one would you be most confident presenting? Why?
